In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from dotenv import load_dotenv

load_dotenv()

In [ ]:
spark = (
    SparkSession.builder
    .appName("uk-property-flip-detector")
    .config("spark.jars", "jars/postgresql-jdbc.jar")
    .getOrCreate()
)

In [ ]:
BASE_YEAR = 1995

df = (
    spark.read
    .format("jdbc")
    .option("url", "jdbc:postgresql://192.168.0.204:5432/land_registry")
    .option("dbtable", "repeat_sale_pairs_regression")
    .option("user", os.environ["PGUSER"])
    .option("password", os.environ["PGPASSWORD"])
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "property_id")
    .option("lowerBound", "1")
    .option("upperBound", "15184630")
    .option("numPartitions", "8")
    .load()
)

pairs = (
    df
    .withColumn(
        # month index - diff in months from base + months elapsed from last
        "prev_period",
        (F.year("prev_date_of_transfer") - BASE_YEAR) * 12
        + (F.month("prev_date_of_transfer") - 1),
    )
    .withColumn(
        "curr_period",
        (F.year("curr_date_of_transfer") - BASE_YEAR) * 12
        + (F.month("curr_date_of_transfer") - 1),
    )
    # cast decimal as double
    .withColumn("log_price_ratio", F.col("log_price_ratio").cast("double"))
)


pairs.select(
    "prev_date_of_transfer", "prev_period",
    "curr_date_of_transfer", "curr_period",
).show(10)

### Parallel JDBC read

Parallelism isn't strictly needed at this scale (12.4M rows fits comfortably on one machine), but the read is partitioned to demonstrate how it would scale.

Spark splis range into 8 slices and sends 8 queries to Postgres, one per partition, each of the form:

    SELECT <columns> FROM repeat_sale_pairs_regression
    WHERE property_id >= X AND property_id < Y

Each executor thread reads its slice concurrently. The bounds only control how the range is divided; they don't filter rows.

`.explain(True)` demonstrates the optimisations done by the Catalyst Optimiser when calling select on the two cols.

In [ ]:
pairs.select("prev_period", "curr_period").explain(True)

In [ ]:
# period index validation
pairs.select(
    F.min("prev_period"), F.max("prev_period"),
    F.min("curr_period"), F.max("curr_period"),
).show()

print("backwards:", pairs.filter(F.col("curr_period") < F.col("prev_period")).count())
print("same month:", pairs.filter(F.col("curr_period") == F.col("prev_period")).count())

Trimming and filtering: drops July 2026 (incomplete registrations) and same-month pairs (zero rows in the design matrix); keeps only regression columns before caching.

In [ ]:
MAX_PERIOD = 377  # July 2026 (378) trimmed: incomplete due to registration lag

pairs_clean = (
    pairs
    .filter(F.col("curr_period") <= MAX_PERIOD)
    .filter(F.col("curr_period") != F.col("prev_period"))
    .select("property_id", "prev_period", "curr_period", "log_price_ratio")
    .cache()
)

print("pairs after filtering:", pairs_clean.count())

print("overlap:", pairs.filter(
    (F.col("curr_period") == 378) & (F.col("curr_period") == F.col("prev_period"))
).count())